In [4]:
from river import datasets, linear_model, tree, metrics
from joblib import Parallel, delayed

# Load a small streaming dataset
dataset = datasets.Phishing()

# Define some candidate models
models = {
    "LogReg": linear_model.LogisticRegression(),
    "HoeffdingTree": tree.HoeffdingTreeClassifier(),
}

# Function to evaluate a single model incrementally
def evaluate_model(name, model, dataset):
    metric = metrics.Accuracy()
    for x, y in dataset:
        y_pred = model.predict_one(x)
        if y_pred is not None:   # only update metric if prediction exists
            metric.update(y, y_pred)
        model.learn_one(x, y)    # ✅ just call it, don't reassign
    return name, metric.get()

# Run evaluations in parallel
results = Parallel(n_jobs=-1)(
    delayed(evaluate_model)(name, model, dataset)
    for name, model in models.items()
)

print("Results:")
for name, score in results:
    print(f"{name}: {score:.4f}")


Results:
LogReg: 0.7304
HoeffdingTree: 0.8799


In [5]:
import pandas as pd
import numpy as np

# Generate synthetic data
N = 50_000_000
df = pd.DataFrame({
    "user_id": np.random.randint(0, 100_000, size=N),
    "event_type": np.random.choice(["click", "view", "purchase"], size=N),
    "value": np.random.rand(N) * 100,
})

# Save to Parquet
df.to_parquet("events.parquet")


In [6]:
import duckdb
import time

con = duckdb.connect()

query = """
SELECT 
    event_type,
    COUNT(*) AS n_events,
    AVG(value) AS avg_value
FROM 'events.parquet'
WHERE value > 50
GROUP BY event_type
"""

start = time.time()
result_duckdb = con.execute(query).fetchdf()
print(result_duckdb)
print("DuckDB time:", time.time() - start, "seconds")


  event_type  n_events  avg_value
0      click   8337933  74.990298
1       view   8329923  75.000989
2   purchase   8333437  75.003566
DuckDB time: 0.5160171985626221 seconds


In [9]:
import dask.dataframe as dd
import time

# Load dataset in Dask
ddf = dd.read_parquet("events.parquet")

start = time.time()
result_dask = (
    ddf[ddf["value"] > 50]
    .groupby("event_type")
    .agg({"value": "mean", "user_id": "count"})
    .compute()
)
print(result_dask)
print("Dask time:", time.time() - start, "seconds")


                value  user_id
event_type                    
purchase    75.003566  8333437
view        75.000989  8329923
click       74.990298  8337933
Dask time: 7.025043725967407 seconds


In [13]:
import duckdb
import dask.dataframe as dd
from dask_ml.linear_model import LogisticRegression

# DuckDB query → Dask DataFrame
con = duckdb.connect()
df = con.execute("""
SELECT value,
       CASE WHEN event_type='click' THEN 1 ELSE 0 END AS is_click,
       CASE WHEN event_type='view' THEN 1 ELSE 0 END AS is_view,
       CASE WHEN event_type='purchase' THEN 1 ELSE 0 END AS is_purchase,
       user_id % 2 AS label
FROM 'events.parquet'
""").fetchdf()

ddf = dd.from_pandas(df, npartitions=8)
X = ddf.drop(columns=["label"])
y = ddf["label"]

# Convert to Dask Arrays (required)
X_arr = X.to_dask_array(lengths=True)
y_arr = y.to_dask_array(lengths=True)

# Train
model = LogisticRegression()
model.fit(X_arr, y_arr)

# Predict
y_pred = model.predict(X_arr)


In [16]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# y_true and y_pred as Dask Arrays → convert to NumPy
y_true = y_arr.compute()
y_pred_vals = y_pred.compute()

# Example metrics
accuracy = accuracy_score(y_true, y_pred_vals)
f1 = f1_score(y_true, y_pred_vals)
print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 0.50006904
F1 Score: 0.5289777480971842


In [17]:
from sklearn.metrics import confusion_matrix, roc_auc_score

cm = confusion_matrix(y_true, y_pred_vals)
auc = roc_auc_score(y_true, y_pred_vals)

print("Confusion Matrix:\n", cm)
print("AUC:", auc)

Confusion Matrix:
 [[10967366 14029551]
 [10966997 14036086]]
AUC: 0.5000614789138382


In [19]:
from dask_ml.metrics import accuracy_score as dask_acc

# y_arr and y_pred are Dask Arrays
acc = dask_acc(y_arr, y_pred)  # returns a scalar (float)
print("Dask Accuracy:", acc)


Dask Accuracy: 0.50006904
